# Khám phá cấu trúc 2 batch dữ liệu Image Retrieval (file .zip trên Google Drive)

**Ý tưởng chính:** ZIP lưu *central directory* (danh mục toàn bộ entry) ở **cuối file**.
Liệt kê cấu trúc chỉ cần đọc vài trăm KB cuối — không cần giải nén, thậm chí không cần
tải cả file về.

## Ba chế độ nguồn (`MODE`)

| MODE | Cách lấy dữ liệu | Khi nào dùng |
|---|---|---|
| `'remote'` | Đọc trực tiếp qua **HTTP Range** từ link Drive, chỉ tải phần cần | **Mặc định nên dùng.** Chỉ muốn xem cấu trúc / xem thử vài ảnh |
| `'gdown'` | `gdown` tải cả file `.zip` về `/content` (có resume) | Cần giải nén để train |
| `'drive'` | Mount Drive, đọc file tại chỗ | Zip nằm trong Drive của chính bạn |

Mọi mục phía sau dùng chung hàm `open_zip(key)` nên **đổi `MODE` là chạy được ngay**,
không phải sửa gì khác.

Trình tự: mục 1 (cấu hình) → 2 (đọc index) → 3 (cây thư mục) → 4 (thống kê & so sánh)
→ 5 (xem trước ảnh/CSV) → 6 (giải nén, tuỳ chọn) → 7 (xuất báo cáo).

## 1. Cấu hình nguồn

In [ ]:
!pip install -q -U gdown requests

In [ ]:
# ============ CAU HINH -- CHI SUA O CELL NAY ============
MODE = 'remote'      # 'remote' | 'gdown' | 'drive'

# --- Cach 1 (tien nhat): dan link FOLDER duoc share, notebook tu tim cac .zip ben trong.
FOLDER = ''   # vd 'https://drive.google.com/drive/folders/<FOLDER_ID>?usp=sharing'

# --- Cach 2: dan link rieng cua TUNG FILE (bo trong FOLDER o tren neu dung cach nay).
# Dan link share hoac file-id cua tung batch (dung cho MODE 'remote' va 'gdown').
# Chap nhan moi dang:
#   https://drive.google.com/file/d/<ID>/view?usp=sharing
#   https://drive.google.com/open?id=<ID>
#   https://drive.google.com/uc?id=<ID>
#   hoac chi dan <ID>
LINKS = {
    'batch1': 'https://drive.google.com/file/d/PASTE_ID_BATCH1/view?usp=sharing',
    'batch2': 'https://drive.google.com/file/d/PASTE_ID_BATCH2/view?usp=sharing',
}

# Chi dung khi MODE == 'drive'
DRIVE_PATHS = {
    'batch1': '/content/drive/MyDrive/batch1.zip',
    'batch2': '/content/drive/MyDrive/batch2.zip',
}

# Chi dung khi MODE == 'gdown': noi luu file tai ve
DOWNLOAD_DIR = '/content/zips'

# LUU Y 1: file/folder phai share 'Anyone with the link' (vai tro Viewer la du).
# LUU Y 2: link FOLDER (/drive/folders/...) KHONG dung truc tiep duoc cho tung batch --
#          endpoint tai cua Drive chi phuc vu file. Hay dat no vao bien FOLDER o tren.
print('MODE =', MODE)

### 1.1 Lớp đọc zip qua HTTP Range

`HTTPFile` giả lập một file-object có `seek`/`read` nhưng mỗi lần đọc chỉ gửi một
`Range` request. `zipfile` chỉ cần seek được, nên nó hoạt động y như file local —
đọc index một zip 25 GB tốn khoảng vài trăm KB truyền tải.

In [ ]:
import io, os, re, html, zipfile
import requests


def extract_file_id(link: str) -> str:
    """Lay file-id tu moi dang link Google Drive (hoac tra lai chinh no neu da la id)."""
    link = link.strip()
    if '/folders/' in link:
        raise ValueError(
            'Day la link FOLDER, khong phai file. Endpoint tai cua Drive chi phuc vu file. '
            'Hay dat link nay vao bien FOLDER o cell cau hinh (notebook se tu liet ke cac '
            '.zip ben trong), hoac copy link rieng cua tung file trong folder.')
    for pat in (r'/file/d/([A-Za-z0-9_-]{10,})',
                r'[?&]id=([A-Za-z0-9_-]{10,})',
                r'/d/([A-Za-z0-9_-]{10,})'):
        m = re.search(pat, link)
        if m:
            return m.group(1)
    if re.fullmatch(r'[A-Za-z0-9_-]{10,}', link):
        return link
    raise ValueError(f'Khong tim thay file-id trong: {link!r}')


def resolve_gdrive(file_id: str):
    """Tra ve (session, direct_url) da vuot qua trang canh bao virus-scan cua Drive."""
    s = requests.Session()
    r = s.get('https://drive.usercontent.google.com/download',
              params={'id': file_id, 'export': 'download', 'confirm': 't'},
              stream=True, allow_redirects=True)
    if 'text/html' in r.headers.get('Content-Type', ''):
        page = r.text
        r.close()
        form = re.search(r'<form[^>]+action="([^"]+)"(.*?)</form>', page, re.S)
        if not form:
            raise RuntimeError(
                'Drive khong tra ve file. Thuong do: file chua share public, '
                'hoac da vuot quota tai xuong. Hay doi sang MODE="drive".')
        action = html.unescape(form.group(1))
        fields = dict(re.findall(r'name="([^"]+)"\s+value="([^"]*)"', form.group(2)))
        r = s.get(action, params=fields, stream=True, allow_redirects=True)
    url = r.url
    r.close()
    return s, url


class HTTPFile(io.RawIOBase):
    """File-object chi doc, seek duoc, lay du lieu bang HTTP Range request."""

    def __init__(self, url, session=None):
        self.url = url
        self.sess = session or requests.Session()
        self._pos = 0
        self.n_requests = 0
        self.n_bytes = 0
        self.size = self._probe()

    def _probe(self) -> int:
        r = self.sess.get(self.url, headers={'Range': 'bytes=0-0'}, stream=True)
        cr = r.headers.get('Content-Range')
        r.close()
        if not cr:
            raise RuntimeError('Server khong ho tro HTTP Range -> dung MODE="gdown".')
        return int(cr.rsplit('/', 1)[-1])

    # -- giao dien file --
    def seekable(self): return True
    def readable(self): return True
    def writable(self): return False
    def tell(self): return self._pos

    def seek(self, offset, whence=io.SEEK_SET):
        if whence == io.SEEK_SET:
            self._pos = offset
        elif whence == io.SEEK_CUR:
            self._pos += offset
        else:
            self._pos = self.size + offset
        self._pos = max(0, min(self._pos, self.size))
        return self._pos

    def read(self, n=-1):
        if n is None or n < 0:
            n = self.size - self._pos
        n = min(n, self.size - self._pos)
        if n <= 0:
            return b''
        end = self._pos + n - 1
        r = self.sess.get(self.url, headers={'Range': f'bytes={self._pos}-{end}'})
        r.raise_for_status()
        data = r.content
        self.n_requests += 1
        self.n_bytes += len(data)
        self._pos += len(data)
        return data

    def readinto(self, buf):
        data = self.read(len(buf))
        buf[:len(data)] = data
        return len(data)


print('HTTPFile san sang')

### 1.2 Link **folder** vs link **file** — và cell chẩn đoán

Đây là chỗ sai phổ biến nhất. Endpoint tải của Drive chỉ phục vụ **nội dung của một file**:

| Link bạn copy được | Dạng | Dùng trực tiếp cho `remote`/`gdown`? |
|---|---|---|
| `drive.google.com/file/d/<ID>/view?usp=sharing` | file | **được** |
| `drive.google.com/drive/folders/<ID>?usp=sharing` | folder | **không** — phải liệt kê file bên trong trước |

Share ở mức *Anyone with the link — Viewer* là **đủ quyền** (Viewer vẫn tải được). Lỗi
không nằm ở vai trò, mà ở việc folder ID không phải file ID: đưa folder ID vào endpoint
tải thì Drive trả về một trang HTML, không phải dữ liệu zip.

`list_drive_folder()` liệt kê file trong folder public mà không cần API key, thử 3 cách
theo thứ tự: `embeddedfolderview` → blob `_DRIVE_ivd` trên trang folder → `gdown`.

`diagnose(link)` in ra **đúng những gì Drive trả về** (HTTP status, Content-Type,
Accept-Ranges, và trích nội dung trang lỗi nếu có) để biết chắc nguyên nhân thay vì đoán.

In [ ]:
import json, re, html
import requests

UA = ('Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 '
      '(KHTML, like Gecko) Chrome/125.0 Safari/537.36')


def classify_link(link: str):
    """-> ('folder'|'file', id). Nhan ca link day du va id tran."""
    link = link.strip()
    m = re.search(r'/folders/([A-Za-z0-9_-]{10,})', link)
    if m:
        return 'folder', m.group(1)
    return 'file', extract_file_id(link)


def list_drive_folder(folder_id: str, session=None, verbose=True):
    """Liet ke file trong mot folder duoc share public. Khong can API key."""
    s = session or requests.Session()
    items, how = [], None

    # -- Cach 1: embeddedfolderview tra ve HTML danh sach rat gon --
    try:
        r = s.get('https://drive.google.com/embeddedfolderview',
                  params={'id': folder_id}, headers={'User-Agent': UA}, timeout=30)
        if getattr(r, 'ok', False):
            for m in re.finditer(
                    r'id="entry-([A-Za-z0-9_-]{20,})".*?flip-entry-title[^>]*>([^<]*)<',
                    r.text, re.S):
                items.append({'id': m.group(1), 'name': html.unescape(m.group(2).strip())})
            if items:
                how = 'embeddedfolderview'
    except Exception as e:
        if verbose:
            print(f'   (embeddedfolderview loi: {type(e).__name__}: {e})')

    # -- Cach 2: blob window['_DRIVE_ivd'] tren trang folder --
    if not items:
        try:
            r = s.get(f'https://drive.google.com/drive/folders/{folder_id}',
                      headers={'User-Agent': UA}, timeout=30)
            m = re.search(r"window\['_DRIVE_ivd'\]\s*=\s*'([^']+)'", r.text)
            if m:
                data = json.loads(m.group(1).encode().decode('unicode_escape'))
                for e in (data[0] or []):
                    items.append({'id': e[0], 'name': e[2],
                                  'mime': e[3] if len(e) > 3 else None})
                if items:
                    how = '_DRIVE_ivd'
        except Exception as e:
            if verbose:
                print(f'   (_DRIVE_ivd loi: {type(e).__name__}: {e})')

    # -- Cach 3: de gdown tu lo --
    if not items:
        try:
            import gdown
            res = gdown.download_folder(id=folder_id, skip_download=True,
                                        quiet=True, remaining_ok=True)
            for f in (res or []):
                items.append({'id': getattr(f, 'id', None),
                              'name': getattr(f, 'local_path', str(f))})
            if items:
                how = 'gdown.download_folder'
        except Exception as e:
            if verbose:
                print(f'   (gdown loi: {type(e).__name__}: {e})')

    seen, out = set(), []
    for it in items:
        if it.get('id') and it['id'] not in seen:
            seen.add(it['id'])
            out.append(it)
    return out, how


def diagnose(link: str, session=None):
    """In ra chinh xac Drive dang tra ve gi cho link nay."""
    s = session or requests.Session()
    print('=' * 78)
    print('LINK:', link)
    try:
        kind, ident = classify_link(link)
    except ValueError as e:
        print('  !! extract_file_id:', e)
        return
    print(f'  nhan dang: {kind}  id={ident}')

    if kind == 'folder':
        items, how = list_drive_folder(ident, s)
        print(f'  liet ke bang: {how} -> {len(items)} muc')
        for it in items:
            print(f'     {it["id"]}  {it.get("name")}')
        zips = [it for it in items
                if str(it.get('name', '')).lower().endswith(('.zip', '.z01', '.tar', '.tar.gz'))]
        if not items:
            print('  >>> Khong liet ke duoc. Folder co the chua share public, hoac Google doi')
            print('      markup. Cach chac chan nhat: mo folder, copy link RIENG tung file.')
        elif zips:
            print(f'  >>> Tim thay {len(zips)} file nen. Dan vao LINKS (hoac dat FOLDER=link nay):')
            for it in zips:
                stem = str(it['name']).rsplit('.', 1)[0]
                print(f"         '{stem}': '{it['id']}',")
        return

    r = s.get('https://drive.usercontent.google.com/download',
              params={'id': ident, 'export': 'download', 'confirm': 't'},
              headers={'User-Agent': UA, 'Range': 'bytes=0-0'}, stream=True, timeout=30)
    h = r.headers
    print(f'  HTTP {r.status_code} | Content-Type: {h.get("Content-Type")}')
    print(f'  Content-Length: {h.get("Content-Length")} | '
          f'Content-Range: {h.get("Content-Range")} | '
          f'Accept-Ranges: {h.get("Accept-Ranges")}')

    if 'text/html' in (h.get('Content-Type') or ''):
        page = r.text
        r.close()
        t = re.search(r'<title>(.*?)</title>', page, re.S)
        print('  Drive tra ve HTML, khong phai du lieu zip. <title>:',
              t.group(1).strip() if t else '?')
        for kw, why in [
            ('quota', 'File public bi tai qua nhieu -> cho vai gio, hoac dung MODE="drive"'),
            ("can't view or download", 'Bi chan tai xuong hoac vuot quota -> MODE="drive"'),
            ('request access', 'Link chua share public (hoac sai id)'),
            ('need permission', 'Link chua share public'),
            ('sign in', 'Drive doi dang nhap -> chua share public'),
            ('virus scan warning', 'Chi la trang canh bao -- resolve_gdrive se vuot duoc'),
        ]:
            if kw in page.lower():
                print('  >>>', why)
        body = ' '.join(re.sub(r'<[^>]+>', ' ', page).split())
        print('  Trich trang:', body[:320])
    else:
        r.close()
        ok_range = h.get('Content-Range') is not None
        print('  >>> Drive tra ve du lieu nhi phan.',
              'Range duoc ho tro -> MODE="remote" chay duoc.' if ok_range
              else 'Nhung KHONG thay Content-Range -> dung MODE="gdown".')


print('classify_link / list_drive_folder / diagnose san sang')

Chạy cell dưới khi gặp lỗi — nó cho biết nguyên nhân thật sự:

In [ ]:
# Chan doan: dan vao bat ky link nao dang co van de
for lk in ([FOLDER] if FOLDER else []) + list(LINKS.values()):
    diagnose(lk)

### 1.3 `open_zip()` — một cửa vào duy nhất cho cả 3 chế độ

In [ ]:
import os, zipfile, io

SOURCES = {}     # key -> dict mo ta nguon
ZIPS = {}        # key -> duong dan local (chi co o mode 'drive'/'gdown')
_REMOTE = {}     # cache (session, url, size) de khong resolve lai moi lan

# Neu dan link FOLDER -> tu dong liet ke cac file nen ben trong thanh LINKS
if FOLDER and MODE in ('remote', 'gdown'):
    _fid = classify_link(FOLDER)[1]
    _items, _how = list_drive_folder(_fid)
    _zips = [it for it in _items
             if str(it.get('name', '')).lower().endswith(('.zip', '.tar', '.tar.gz'))]
    print(f'Folder {_fid} (qua {_how}): {len(_items)} muc, {len(_zips)} file nen')
    if not _zips:
        raise RuntimeError(
            'Khong liet ke duoc file nen trong folder. Chay diagnose(FOLDER) de xem ly do, '
            'hoac copy link rieng cua tung file vao LINKS.')
    LINKS = {str(it['name']).rsplit('.', 1)[0]: it['id']
             for it in sorted(_zips, key=lambda x: str(x['name']))}
    for _k, _v in LINKS.items():
        print(f'   {_k:20s} {_v}')
    print('>>> Cac key batch hien tai:', list(LINKS))

if MODE == 'drive':
    from google.colab import drive
    drive.mount('/content/drive')
    for k, p in DRIVE_PATHS.items():
        SOURCES[k] = {'kind': 'local', 'path': p}
        ZIPS[k] = p

elif MODE == 'gdown':
    import gdown
    os.makedirs(DOWNLOAD_DIR, exist_ok=True)
    for k, link in LINKS.items():
        fid = extract_file_id(link)
        out = os.path.join(DOWNLOAD_DIR, f'{k}.zip')
        if os.path.exists(out):
            print(f'[{k}] da co san: {out} ({os.path.getsize(out) / 1e9:.2f} GB) -- bo qua tai')
        else:
            print(f'[{k}] gdown {fid} -> {out}')
            gdown.download(id=fid, output=out, quiet=False, resume=True)
        SOURCES[k] = {'kind': 'local', 'path': out}
        ZIPS[k] = out

elif MODE == 'remote':
    for k, link in LINKS.items():
        fid = extract_file_id(link)
        sess, url = resolve_gdrive(fid)
        SOURCES[k] = {'kind': 'remote', 'id': fid, 'session': sess, 'url': url}

else:
    raise ValueError(f'MODE khong hop le: {MODE!r}')


def open_zip(key, buffer_size=1 << 20):
    """Tra ve zipfile.ZipFile cho batch `key`, bat ke nguon la local hay remote."""
    src = SOURCES[key]
    if src['kind'] == 'local':
        return zipfile.ZipFile(src['path'])
    raw = HTTPFile(src['url'], session=src['session'])
    zf = zipfile.ZipFile(io.BufferedReader(raw, buffer_size=buffer_size))
    zf._httpfile = raw      # giu lai de xem so request / so byte da tai
    return zf


# Kiem tra nguon
for k in SOURCES:
    src = SOURCES[k]
    if src['kind'] == 'local':
        ok = os.path.exists(src['path'])
        sz = f'{os.path.getsize(src["path"]) / 1e9:.2f} GB' if ok else '---'
        print(f'{k:8s} local  {"OK" if ok else "MISSING":8s} {sz:>10s}  {src["path"]}')
    else:
        probe = HTTPFile(src['url'], session=src['session'])
        print(f'{k:8s} remote OK       {probe.size / 1e9:.2f} GB  id={src["id"]}')

Nếu `resolve_gdrive` báo lỗi: kiểm tra file đã share **Anyone with the link** chưa.
Drive cũng chặn tạm khi một file public bị tải quá nhiều — khi đó dùng `MODE='drive'`
(thêm shortcut file vào Drive của bạn rồi mount).

Link trực tiếp mà `resolve_gdrive` trả về gắn với session và sẽ hết hạn sau một thời gian.
Nếu đang chạy lâu mà các request bắt đầu lỗi, chạy lại cell 1.1 + 1.3 để lấy link mới —
`INDEX` đã đọc rồi nên không mất gì.

Cell dưới quét zip trong Drive, chỉ hữu ích với `MODE='drive'`:

In [ ]:
import os, glob

if MODE == 'drive':
    ROOT = '/content/drive/MyDrive'
    found = set()
    for depth in range(1, 5):
        found |= set(glob.glob(os.path.join(ROOT, *(['*'] * depth)) + '.zip'))
    for p in sorted(found):
        print(f'{os.path.getsize(p) / 1e9:8.2f} GB  {p}')
    print(f'\nTong: {len(found)} file zip')
else:
    print(f'Bo qua (MODE={MODE!r})')

## 2. Đọc index của zip (không giải nén)

Với `MODE='remote'`, cell này in luôn số HTTP request và số byte thực sự đã tải — thường
chỉ vài trăm KB cho zip hàng chục GB. Kết quả cache trong `INDEX`, các mục sau chạy tức thì.

In [ ]:
import os, time
import pandas as pd

def read_zip_index(key: str) -> pd.DataFrame:
    """Doc toan bo danh sach entry trong zip ma KHONG giai nen."""
    rows = []
    with open_zip(key) as z:
        for info in z.infolist():
            clean = info.filename.rstrip('/')
            rows.append({
                'path': info.filename,
                'is_dir': info.is_dir(),
                'size': info.file_size,
                'csize': info.compress_size,
                'crc': info.CRC,
                'depth': clean.count('/'),
                'ext': os.path.splitext(clean)[1].lower(),
                'name': os.path.basename(clean),
                'parent': os.path.dirname(clean),
            })
        stats = getattr(z, '_httpfile', None)
        net = (f' | {stats.n_requests} HTTP request, '
               f'{stats.n_bytes / 1e6:.2f} MB da tai') if stats else ''
    df = pd.DataFrame(rows)
    df.attrs['net'] = net
    return df

INDEX = {}
for k in SOURCES:
    t0 = time.time()
    INDEX[k] = read_zip_index(k)
    files = INDEX[k][~INDEX[k].is_dir]
    print(f'[{k}] {len(INDEX[k]):,} entries | {len(files):,} files | '
          f'{files["size"].sum() / 1e9:.2f} GB sau giai nen | '
          f'{time.time() - t0:.1f}s{INDEX[k].attrs["net"]}')

## 3. Cây thư mục

In [ ]:
from collections import defaultdict

def _human(n):
    for u in ['B', 'KB', 'MB', 'GB', 'TB']:
        if n < 1024:
            return f'{n:.1f}{u}'
        n /= 1024
    return f'{n:.1f}PB'

def print_tree(df, max_depth=3, max_children=12, file_samples=2):
    """In cay thu muc: moi folder kem so file (de quy) va dung luong."""
    files = df[~df.is_dir]
    n_files, tot_size = defaultdict(int), defaultdict(int)
    children = defaultdict(set)

    for path, size in zip(files['path'], files['size']):
        parts = path.split('/')
        n_files[''] += 1
        tot_size[''] += size
        for i, part in enumerate(parts):
            prefix = '/'.join(parts[:i])
            full = '/'.join(parts[:i + 1])
            is_last = (i == len(parts) - 1)
            children[prefix].add(part if is_last else part + '/')
            if not is_last:
                n_files[full] += 1
                tot_size[full] += size

    def walk(prefix, depth, indent):
        kids = sorted(children.get(prefix, ()))
        dirs = [k for k in kids if k.endswith('/')]
        plain = [k for k in kids if not k.endswith('/')]

        for d in dirs[:max_children]:
            full = f'{prefix}/{d[:-1]}' if prefix else d[:-1]
            print(f'{indent}|-- {d:<38s} {n_files[full]:>9,} files  {_human(tot_size[full]):>9s}')
            if depth < max_depth:
                walk(full, depth + 1, indent + '|   ')
            elif children.get(full):
                kk = children[full]
                nd = sum(1 for x in kk if x.endswith('/'))
                print(f'{indent}|       ... {nd} folder + {len(kk) - nd} file (tang sau an)')
        if len(dirs) > max_children:
            print(f'{indent}|-- ... va {len(dirs) - max_children} thu muc khac')

        for f in plain[:file_samples]:
            print(f'{indent}|-- {f}')
        if len(plain) > file_samples:
            print(f'{indent}|-- ... va {len(plain) - file_samples} file khac')

    print(f'ROOT   {n_files[""]:,} files   {_human(tot_size[""])}')
    walk('', 1, '')


for k in INDEX:
    print('=' * 92)
    print(f'### {k}')
    print('=' * 92)
    print_tree(INDEX[k], max_depth=3)
    print()

In [ ]:
# Zoom vao mot nhanh cu the.
# Luu y: startswith -> PREFIX='batch1/train' cung khop 'batch1/train.csv';
# them dau '/' o cuoi neu muon dung folder do thoi.
K = list(INDEX)[0]          # doi thanh key batch khac neu muon
PREFIX = ''                 # vd: 'batch1/train/images'

sub = INDEX[K]
sub = sub[sub.path.str.startswith(PREFIX)]
print(f'{K} / {PREFIX!r}: {len(sub[~sub.is_dir]):,} files\n')
print_tree(sub, max_depth=6, max_children=25)

## 4. Thống kê & so sánh 2 batch

In [ ]:
def summary(df, name):
    files = df[~df.is_dir]
    print(f'=== {name} ===')
    print(f'  so file        : {len(files):,}')
    print(f'  so thu muc     : {int(df.is_dir.sum()):,}')
    print(f'  dung luong goc : {files["size"].sum() / 1e9:.2f} GB')
    print(f'  dung luong nen : {files["csize"].sum() / 1e9:.2f} GB')
    print(f'  do sau toi da  : {int(files["depth"].max())}')

    print('\n  -- Phan mo rong --')
    ext = (files.groupby('ext')
                .agg(n=('path', 'size'), GB=('size', lambda s: round(s.sum() / 1e9, 3)))
                .sort_values('n', ascending=False))
    print(ext.head(15).to_string())

    print('\n  -- Thu muc cap 1 --')
    print(files['path'].str.split('/').str[0].value_counts().head(15).to_string())

    print('\n  -- 10 folder nhieu file nhat --')
    print(files['parent'].value_counts().head(10).to_string())
    print()

for k in INDEX:
    summary(INDEX[k], k)

In [ ]:
# So sanh cau truc 2 batch theo tung cap
ka, kb = list(INDEX)[:2]
a, b = INDEX[ka], INDEX[kb]

def dir_set(df, levels):
    s = df[~df.is_dir]['path'].str.split('/')
    return set(s.str[:levels].str.join('/').unique())

for lv in (1, 2, 3):
    A, B = dir_set(a, lv), dir_set(b, lv)
    print(f'--- Cau truc den cap {lv} ---')
    print(f'  chi co o {ka:8s}: {len(A - B):6,d}  {sorted(A - B)[:6]}')
    print(f'  chi co o {kb:8s}: {len(B - A):6,d}  {sorted(B - A)[:6]}')
    print(f'  chung            : {len(A & B):6,d}  {sorted(A & B)[:6]}')
    print()

In [ ]:
# Trung lap giua 2 batch -- quan trong voi image retrieval (tranh leak gallery/query).
# CRC32 da nam san trong index -> so sanh noi dung ma khong can giai nen.
fa = a[~a.is_dir]
fb = b[~b.is_dir]

na, nb_ = set(fa['name']), set(fb['name'])
print(f'Trung TEN file       : {len(na & nb_):,}   vd: {sorted(na & nb_)[:8]}')

ca, cb = set(fa['crc']), set(fb['crc'])
print(f'Trung NOI DUNG (CRC32): {len(ca & cb):,}')
print(f'  {ka}: {len(ca):,} noi dung duy nhat / {len(fa):,} file')
print(f'  {kb}: {len(cb):,} noi dung duy nhat / {len(fb):,} file')

dup = fa[fa['crc'].isin(ca & cb)]['path'].head(10).tolist()
print(f'  vd file trung noi dung o {ka}: {dup}')

## 5. Xem trước nội dung — streaming trực tiếp từ zip

`z.open(entry)` chỉ giải nén đúng entry đó trong RAM. Ở `MODE='remote'` nó chỉ tải đúng
dải byte của entry đó, nên xem 8 ảnh thật trong một zip 25 GB chỉ tốn vài MB.

In [ ]:
# Liet ke file metadata de biet nhan / annotation nam o dau
META_EXT = {'.csv', '.tsv', '.json', '.jsonl', '.txt', '.yaml', '.yml', '.md', '.xml'}

for k in INDEX:
    f = INDEX[k]
    m = f[(~f.is_dir) & f.ext.isin(META_EXT)]
    print(f'--- {k}: {len(m)} file metadata ---')
    print(m[['path', 'size']].head(30).to_string(index=False))
    print()

In [ ]:
# Doc 1 file CSV / JSON / text ngay trong zip
import json
import pandas as pd

K = list(INDEX)[0]
ENTRY = INDEX[K][(~INDEX[K].is_dir) & INDEX[K].ext.isin(META_EXT)]['path'].iloc[0]
# ENTRY = 'batch1/train.csv'   # >>> hoac dien tay theo ket qua cell tren <<<
print('Doc:', ENTRY)

with open_zip(K) as z:
    if ENTRY.endswith(('.csv', '.tsv')):
        sep = '\t' if ENTRY.endswith('.tsv') else ','
        with z.open(ENTRY) as fh:
            df = pd.read_csv(fh, sep=sep)
        print('shape:', df.shape)
        print(df.dtypes.to_string())
        display(df.head(20))
    elif ENTRY.endswith(('.json', '.jsonl')):
        with z.open(ENTRY) as fh:
            raw = fh.read()
        if ENTRY.endswith('.jsonl'):
            lines = raw.decode('utf-8', 'replace').splitlines()
            print(f'{len(lines)} dong; 3 dong dau:')
            for l in lines[:3]:
                print(' ', json.loads(l))
        else:
            obj = json.loads(raw)
            print(type(obj).__name__, '->', list(obj)[:20] if isinstance(obj, dict) else obj[:3])
    else:
        with z.open(ENTRY) as fh:
            print(fh.read(4000).decode('utf-8', 'replace'))

In [ ]:
# Xem truoc anh ngau nhien (khong ghi ra disk)
import io, random
from PIL import Image
import matplotlib.pyplot as plt

IMG_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tif', '.tiff'}
N_SHOW = 8
random.seed(0)

for k in INDEX:
    f = INDEX[k]
    imgs = f[(~f.is_dir) & f.ext.isin(IMG_EXT)]['path'].tolist()
    if not imgs:
        print(f'{k}: khong tim thay anh')
        continue
    pick = random.sample(imgs, min(N_SHOW, len(imgs)))

    fig, axes = plt.subplots(1, len(pick), figsize=(2.6 * len(pick), 3.2))
    axes = [axes] if len(pick) == 1 else list(axes)
    with open_zip(k) as z:
        for ax, name in zip(axes, pick):
            with z.open(name) as fh:
                im = Image.open(io.BytesIO(fh.read()))
                im.load()
            ax.imshow(im)
            ax.set_title(f'{im.size[0]}x{im.size[1]} {im.mode}\n{name.split("/")[-1][:20]}',
                         fontsize=8)
            ax.axis('off')
        stats = getattr(z, '_httpfile', None)
    fig.suptitle(k + (f'  ({stats.n_bytes / 1e6:.1f} MB tai qua mang)' if stats else ''))
    plt.tight_layout()
    plt.show()

In [ ]:
# Phan bo kich thuoc anh (sample nho)
import io, random
from PIL import Image
import pandas as pd

SAMPLE = 100 if MODE == 'remote' else 300   # remote: moi anh la 1 request, lay it thoi
random.seed(0)

for k in INDEX:
    f = INDEX[k]
    imgs = f[(~f.is_dir) & f.ext.isin(IMG_EXT)]['path'].tolist()
    if not imgs:
        continue
    pick = random.sample(imgs, min(SAMPLE, len(imgs)))
    sizes, modes = [], []
    with open_zip(k) as z:
        for name in pick:
            with z.open(name) as fh:
                im = Image.open(io.BytesIO(fh.read()))
            sizes.append(f'{im.size[0]}x{im.size[1]}')
            modes.append(im.mode)
    print(f'--- {k} (sample {len(pick)}) ---')
    print('  WxH pho bien:')
    print(pd.Series(sizes).value_counts().head(8).to_string())
    print(f'  mode: {pd.Series(modes).value_counts().to_dict()}')
    print()

## 6. (Tuỳ chọn) Giải nén

Chỉ chạy khi thật sự cần dữ liệu để train.

- Giải nén vào `/content` (SSD local của Colab), **không** vào Drive: ghi hàng vạn file nhỏ
  qua Drive FUSE rất chậm và dễ timeout.
- Muốn giải nén toàn bộ thì nên `MODE='gdown'` (tải một lần rồi `unzip`) — nhanh hơn nhiều
  so với rút từng file qua HTTP Range.
- `MODE='remote'` vẫn giải nén được (cell 6a) nhưng phù hợp với vài nghìn file trở xuống.

In [ ]:
!df -h /content | tail -1

In [ ]:
# 6a. Giai nen CHON LOC theo prefix / ext / so luong -- chay voi moi MODE
import os, time

K = list(INDEX)[0]
DEST = f'/content/data/{K}'
PREFIX = ''          # vd 'batch1/train/' -> chi lay nhanh nay; '' = tat ca
ONLY_EXT = None      # vd {'.jpg', '.csv'}; None = moi loai
LIMIT = 500          # None = khong gioi han. O MODE remote nen de nho.

os.makedirs(DEST, exist_ok=True)
f = INDEX[K]
sel = f[(~f.is_dir) & f.path.str.startswith(PREFIX)]
if ONLY_EXT:
    sel = sel[sel.ext.isin(ONLY_EXT)]
sel = sel.iloc[:LIMIT] if LIMIT else sel
names = sel['path'].tolist()
print(f'Se giai nen {len(names):,} file, ~{sel["size"].sum() / 1e9:.2f} GB -> {DEST}')

t0 = time.time()
with open_zip(K) as z:
    for i, n in enumerate(names, 1):
        z.extract(n, DEST)
        if i % 500 == 0:
            print(f'  {i:,}/{len(names):,}  ({time.time() - t0:.0f}s)')
print(f'Xong trong {time.time() - t0:.0f}s')

In [ ]:
# 6b. Giai nen TOAN BO -- chi khi MODE in ('gdown', 'drive'), dung `unzip` cua he thong
import os

assert MODE in ('gdown', 'drive'), "Can MODE='gdown' hoac 'drive' de giai nen toan bo"

for k, p in ZIPS.items():
    dest = f'/content/data/{k}'
    os.makedirs(dest, exist_ok=True)
    print(f'>>> unzip {k} -> {dest}')
    !unzip -q -o "{p}" -d "{dest}" && echo "   DONE {k}"

In [ ]:
# Kiem tra sau khi giai nen
!echo '--- so file ---'    && find /content/data -type f | wc -l
!echo '--- cay 3 cap ---'  && find /content/data -maxdepth 3 -type d | head -60
!echo '--- dung luong ---' && du -sh /content/data/* 2>/dev/null

## 7. Xuất báo cáo cấu trúc

Lưu index + cây thư mục để lần sau xem lại ngay, không phải đọc lại zip. Mặc định ghi vào
`/content` rồi tải về; nếu đã mount Drive thì đổi `OUT` sang đường dẫn trong Drive.

In [ ]:
import os, io, contextlib

OUT = '/content/ir_zip_report'
# OUT = '/content/drive/MyDrive/ir_zip_report'   # dung neu da mount Drive
os.makedirs(OUT, exist_ok=True)

for k in INDEX:
    INDEX[k].to_csv(f'{OUT}/{k}_index.csv.gz', index=False, compression='gzip')
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        print_tree(INDEX[k], max_depth=4, max_children=40)
    with open(f'{OUT}/{k}_tree.txt', 'w', encoding='utf-8') as fh:
        fh.write(buf.getvalue())
    print(f'{k}: da luu index + tree')

!ls -lh "{OUT}"

In [ ]:
# Tai bao cao ve may (nho, vai MB)
import shutil
from google.colab import files

shutil.make_archive('/content/ir_zip_report', 'zip', OUT)
files.download('/content/ir_zip_report.zip')